# Tenuto — Expressive Score-to-Performance AI Engine

Predicts human performance nuance (rubato, micro-timing, velocity, articulation, sustain pedal) from sheet music or MIDI scores.

### Stage 1: Setup Repository & Environment
**Note:** To enable TPU acceleration in Google Colab, go to **Runtime > Change runtime type > TPU v5e-1** (or **T4 GPU**).

In [ ]:
# 1. Clone repo if needed or pull latest changes
import os
if not os.path.exists("/content/tenuto"):
    get_ipython().system("git clone https://github.com/kyleconciso/tenuto.git /content/tenuto")
else:
    get_ipython().system("cd /content/tenuto && git pull")

# 2. Set working directory & PYTHONPATH globally
%cd /content/tenuto
%env PYTHONPATH=/content/tenuto:.

# 3. Install dependencies & verify PyTorch
import torch
print("PyTorch:", torch.__version__)
%pip install -q partitura mido scipy tqdm matplotlib huggingface_hub midi2audio pandas pyarrow rclone jax flax optax


### Stage 2: Connect Google Drive & Sync Preprocessed Dataset
Mount your Google Drive to automatically load or save dataset archives and model checkpoints.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 2. Idempotent check & extract for preprocessed dataset
import os, zipfile, glob, shutil
target_processed = "/content/tenuto/data/processed"
os.makedirs(target_processed, exist_ok=True)

pt_files = glob.glob(os.path.join(target_processed, "**/*.pt"), recursive=True)
if len(pt_files) >= 1000:
    print(f"[TenutoColab] ✅ Preprocessed dataset already extracted ({len(pt_files)} files found). Skipping extraction!")
else:
    gdrive_zips = [
        "/content/drive/MyDrive/Tenuto/processed.zip",
        "/content/drive/MyDrive/Tenuto/storage.zip",
        "/content/drive/MyDrive/processed.zip"
    ]
    zip_path = None
    for zp in gdrive_zips:
        if os.path.exists(zp):
            zip_path = zp
            break
            
    if zip_path:
        print(f"[TenutoColab] Extracting preprocessed zip from Google Drive: {zip_path}...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target_processed)
            
        # Move any nested train/val folders up to target_processed if needed
        for root, dirs, files in os.walk(target_processed):
            if root != target_processed and (os.path.basename(root) in ["train", "val"]):
                parent = os.path.dirname(root)
                if parent != target_processed:
                    dst_sub = os.path.join(target_processed, os.path.basename(root))
                    os.makedirs(dst_sub, exist_ok=True)
                    for f in files:
                        shutil.move(os.path.join(root, f), os.path.join(dst_sub, f))
                        
        pt_files_count = len(glob.glob(os.path.join(target_processed, "**/*.pt"), recursive=True))
        print(f"[TenutoColab] 🎉 Extraction complete! ({pt_files_count} .pt files ready in {target_processed})")
    else:
        print("[TenutoColab] ⚠️ No preprocessed zip found in Google Drive. Synthetic sequences will verify training.")


### Stage 3: Preprocess Dataset (Skipped automatically if preprocessed files exist)

In [ ]:
%cd /content/tenuto
import glob
pt_files = glob.glob("/content/tenuto/data/processed/**/*.pt", recursive=True)
if len(pt_files) >= 1000:
    print(f"[TenutoColab] ✅ Preprocessed dataset ready ({len(pt_files)} files). Skipping preprocessing step!")
else:
    get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.download_dataset --dataset combined --pianocore_subset PianoCoRe-A*")
    get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.preprocess --data_dir ./data --processed_dir ./data/processed --max_samples 10000")


### Stage 4: Train Transformer Backbone (Auto TPU v5e-1 Acceleration)
Trains the non-autoregressive score-to-performance Transformer model. Auto-resumes and auto-saves checkpoints per epoch.

In [ ]:
%cd /content/tenuto
import sys
sys.path.insert(0, "/content/tenuto")

# Automatically uses native JAX / Flax on TPU or PyTorch fallback
try:
    import jax
    backend = jax.default_backend()
    print(f"[TenutoColab] 🚀 Running Native JAX / Flax Training on {backend.upper()} ({jax.devices()})...")
    from src.train_jax import main as train_jax_main
    train_jax_main(["--data_dir", "./data/processed", "--in_features", "40", "--epochs", "20", "--batch_size", "16"])
except Exception as e:
    print(f"[TenutoColab] Fallback to PyTorch Training... ({e})")
    from src.train import main as train_py_main
    train_py_main(["--model_type", "transformer", "--data_dir", "./data/processed", "--in_features", "40", "--epochs", "20", "--batch_size", "16"])


### Stage 5: Expressive Inference

In [ ]:
%cd /content/tenuto
get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.infer --score data/asap/Balakirev/Islamey/xml_score.musicxml --checkpoint checkpoints/best_transformer_model.pth --model_type transformer --output_midi output_expressive.mid")


### Stage 6: Listenable Audio Comparison 🎧

In [ ]:
%cd /content/tenuto
get_ipython().system("apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm timidity")

import sys
sys.path.insert(0, "/content/tenuto")
from src.audio import play_audio_in_colab

print("🎵 1. Playing Original Flat Score (Mechanical):")
play_audio_in_colab("data/asap/Balakirev/Islamey/midi_score.mid", title="Original Flat Score")

print("\n🎵 2. Playing Original Human Performance (Ground Truth):")
play_audio_in_colab("data/asap/Balakirev/Islamey/CHEN04.mid", title="Human Performance")

print("\n🎵 3. Playing Tenuto AI Generated Performance:")
play_audio_in_colab("output_expressive.mid", title="Tenuto AI Expressive Performance")
